# Extra credit 공통 분할 파일 생성

extra credit 특징의 라벨을 확인하고 모델팀 공통 분할 `splits/extra_split_seed42.csv`를 만든다.

- 분할 단위는 방문(`source_group_id`)이다. 같은 방문의 split 5개는 모두 train 또는 모두 test에 들어간다.
- extra credit 방문은 메인 실험 pkl의 기록과 같은 방문이다(3장에서 확인). 그래서 메인 `splits/split_seed42.csv`의 train/test를 그대로 따른다. 따로 나누면 메인에서 학습한 방문이 extra test에 들어간다.
- 메인에 없는 unmon 방문만 따로 80:20으로 나눈다. train 방문에는 5-fold 번호를 붙인다.

필요한 파일: `data/extra/features_extra_split_trim0.csv.gz`, `data/extra/features_extra_join_trim0.csv.gz` (루트 README의 데이터 준비 참고). CPU로 몇 분이면 끝난다.

## 0. 설정

In [ ]:
import os, sys, subprocess
from pathlib import Path

def find_repo_root():
    """저장소 루트(splits/ 폴더가 있는 위치)를 찾는다. Colab에서 노트북만 연 경우 저장소를 내려받는다."""
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / "splits" / "split_seed42.csv").exists():
            return p
    if "google.colab" in sys.modules:
        dst = Path("/content/website-fingerprinting")
        if not dst.exists():
            subprocess.run(["git", "clone", "-q", "https://github.com/EWHA-MLB/website-fingerprinting.git", str(dst)], check=True)
        return dst
    raise FileNotFoundError("저장소 안에서 실행해야 한다 (splits/ 폴더를 찾지 못함)")

ROOT       = find_repo_root()
EXTRA_DIR  = ROOT / "data" / "extra"
SPLIT_FILE = EXTRA_DIR / "features_extra_split_trim0.csv.gz"
JOIN_FILE  = EXTRA_DIR / "features_extra_join_trim0.csv.gz"
MAIN_DIR   = ROOT / "features" / "기본 실험"
SPLIT_MAIN = ROOT / "splits" / "split_seed42.csv"
OUT_PATH   = ROOT / "splits" / "extra_split_seed42.csv"

SEED, TEST_SIZE, N_FOLDS = 42, 0.2, 5
for p in [SPLIT_FILE, JOIN_FILE, MAIN_DIR / "features_mon_trim0.csv", SPLIT_MAIN]:
    print("있음" if p.exists() else "없음", "|", p)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

## 1. 데이터 읽기

분할에는 메타데이터와 패킷 수만 필요하다. 빈 split 제외 버전은 `status == "empty"` 행을 뺀 것과 같다.

In [ ]:
META = ["record_id", "source_group_id", "variant", "status", "dataset", "page_id", "sample_idx", "label", "trim"]
CNT  = ["g1_n_total", "g1_n_out"]

inc = pd.read_csv(SPLIT_FILE, usecols=META)
exc = inc[inc.status != "empty"]
jn  = pd.read_csv(JOIN_FILE, usecols=["source_group_id", "dataset", "page_id", "sample_idx", "label"] + CNT)
main_mon   = pd.read_csv(MAIN_DIR / "features_mon_trim0.csv", usecols=["record_id", "page_id", "label"] + CNT)
main_unmon = pd.read_csv(MAIN_DIR / "features_unmon10k_trim0.csv", usecols=["record_id"] + CNT)
main_split = pd.read_csv(SPLIT_MAIN).set_index("record_id").split
print(f"split 포함 {len(inc):,}행 | 제외 {len(exc):,}행 | join {len(jn):,}행")

## 2. 구조·라벨 확인

In [ ]:
def check(name, ok, detail=""):
    print(f"[{'통과' if ok else '실패'}] {name}" + (f" | {detail}" if detail else ""))
    assert ok, name

g = inc.groupby("source_group_id")
check("포함 버전 190,000행 / 제외 버전 172,085행", len(inc) == 190_000 and len(exc) == 172_085)
check("방문 38,000개, 방문마다 split_0~4", g.ngroups == 38_000 and (g.variant.nunique() == 5).all() and (g.size() == 5).all())
check("방문 안에서 dataset·label 동일", (g.dataset.nunique() == 1).all() and (g.label.nunique() == 1).all())
check("split 5개가 모두 빈 방문 없음", exc.source_group_id.nunique() == 38_000)

visits = g.agg(dataset=("dataset", "first"), label=("label", "first")).reset_index()
mon_v, unmon_v = visits[visits.dataset == "mon"], visits[visits.dataset == "unmon"]
per_site = mon_v.label.value_counts()
check("mon 19,000 / unmon 19,000 방문", len(mon_v) == 19_000 and len(unmon_v) == 19_000)
check("mon label 0~94, 사이트당 200방문", sorted(per_site.index) == list(range(95)) and (per_site == 200).all())
check("unmon label = -1", (unmon_v.label == -1).all())
check("extra mon은 label = page_id", (jn[jn.dataset == "mon"].label == jn[jn.dataset == "mon"].page_id).all())

## 3. 메인 기록과의 대응

extra의 `page_id`는 사이트 번호(0~94), `sample_idx`는 0~199다. 메인은 사이트마다 page가 10개(`page_id` 0~949), page마다 sample이 20개다.

extra (사이트 p, sample s) = 메인 (page p×10 + s÷20의 몫, sample s÷20의 나머지)

이 대응으로 짝지은 메인 기록과 총 패킷 수·송신 패킷 수가 모두 같은지 확인한다. 기준선으로 일부러 다른 사이트의 메인 기록과도 비교한다.

In [ ]:
jm = jn[jn.dataset == "mon"].copy()
jm["main_id"] = [f"mon_p{p * 10 + s // 20:03d}_s{s % 20:02d}" for p, s in zip(jm.page_id, jm.sample_idx)]
jm = jm.merge(main_mon, left_on="main_id", right_on="record_id", how="left", suffixes=("", "_m"))
same = (jm.g1_n_total == jm.g1_n_total_m) & (jm.g1_n_out == jm.g1_n_out_m)

key = set(map(tuple, main_mon[["label"] + CNT].to_numpy()))
shift = jm[["label"] + CNT].copy(); shift["label"] = (shift.label + 1) % 95
base = np.mean([tuple(r) in key for r in shift.to_numpy()])
print(f"mon: 대응 규칙으로 패킷 수까지 일치 {same.mean():.4f} ({same.sum():,}/{len(jm):,}) | 다른 사이트 기준선 {base:.4f}")
check("mon 19,000개 모두 메인 기록과 대응", same.all())
jm["split"] = jm.main_id.map(main_split)

## 4. 분할 생성

- mon: 대응하는 메인 기록의 train/test를 따른다.
- unmon: 패킷 수(총·송신)가 같은 메인 unmon 기록이 있으면 그 기록의 train/test를 따른다. 후보가 train과 test에 모두 있으면 test 누수를 막기 위해 train으로 둔다. 후보가 없으면 메인에 없는 방문으로 보고 따로 80:20으로 나눈다.
- fold: train 방문에 `StratifiedKFold(5, shuffle=True, random_state=42)`를 `label` 기준으로 적용한다. test는 -1.

In [ ]:
ju = jn[jn.dataset == "unmon"][["source_group_id"] + CNT]
cand = ju.merge(main_unmon, on=CNT, how="left")
cand["split_m"] = cand.record_id.map(main_split)
cand_splits = cand.groupby("source_group_id").split_m.agg(lambda s: set(s.dropna()))

def decide(s):
    if not s:
        return "new"
    return "test" if s == {"test"} else "train"

u_split = cand_splits.map(decide)
print("unmon:", u_split.value_counts().to_dict(), f"| 후보가 train·test에 걸친 방문 {int((cand_splits.map(len) > 1).sum())}개 → train")

new_ids = u_split[u_split == "new"].index.sort_values()
tr_new, te_new = train_test_split(new_ids, test_size=TEST_SIZE, random_state=SEED)
u_split.loc[tr_new] = "train"
u_split.loc[te_new] = "test"

split = visits.sort_values("source_group_id").reset_index(drop=True)
split["split"] = split.source_group_id.map(pd.concat([jm.set_index("source_group_id").split, u_split]))
split["source"] = np.where(split.source_group_id.isin(new_ids), "new", "main_matched")
assert split.split.isin(["train", "test"]).all()

split["fold"] = -1
tr = split[split.split == "train"]
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for k, (_, va) in enumerate(skf.split(tr, tr.label)):
    split.loc[tr.index[va], "fold"] = k

display(pd.crosstab([split.dataset, split.source], split.split, margins=True))
per = split[split.dataset == "mon"].groupby("label").split.value_counts().unstack()
print(f"사이트당 방문: train {per.train.min()}~{per.train.max()}, test {per.test.min()}~{per.test.max()}")

## 5. 저장

저장소에 이미 분할 파일이 있으면 새로 만든 것과 같은지 먼저 비교한다. 같으면 그대로 두고, 다르면 차이를 출력하고 저장하지 않는다.

In [ ]:
if OUT_PATH.exists():
    old = pd.read_csv(OUT_PATH)
    same_file = old.equals(split[old.columns.tolist()]) if list(old.columns) == list(split.columns) else False
    print("기존 분할 파일과 동일" if same_file else "기존 분할 파일과 다름 → 저장하지 않음. 차이를 확인할 것")
    if not same_file:
        diff = old.merge(split, on="source_group_id", suffixes=("_old", "_new"))
        display(diff[diff.split_old != diff.split_new].head())
else:
    split.to_csv(OUT_PATH, index=False)
    print("저장:", OUT_PATH, f"({len(split):,}행)")